# HW1 - Rethinking Generalization

**Reading:** Zhang, Bengio, Hardt, Recht & Vinyals, *Understanding Deep Learning Requires
Rethinking Generalization*, ICLR 2017 ([arXiv:1611.03530](https://arxiv.org/abs/1611.03530)),
Sections 1-4.

In this assignment you build a small PyTorch training stack and use it to reproduce the paper's
central experiment. The architecture, the optimizer, and the hyperparameters stay fixed
throughout; the only thing that changes is the *data*. Generalization collapses, optimization
does not.

**Instructions.** Fill in every cell marked `TODO` and answer every question marked **Q** in the
markdown cell below it. Do not change the given function signatures - later cells depend on them.

**Compute.** 24 short training runs, ~15 minutes on a recent GPU and proportionally more on
an older one. Every run is cached in `results/`, so re-executing a cell after a kernel restart is
free. Start early.

## Part 0 - Setup

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import hw1_utils as U

DEVICE = U.get_device()
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)
N_TRAIN, N_TEST, EPOCHS = 10_000, 2_000, 60
U.set_seed(0)
print(DEVICE)

In [ ]:
Xtr, ytr, Xte, yte = U.load_cifar10(N_TRAIN, N_TEST)
print(Xtr.shape, ytr.shape, Xte.shape, yte.shape)
print(f"pixel mean {Xtr.mean():.3f}, std {Xtr.std():.3f}")

Images are standardized per color channel using the CIFAR-10 statistics, so the training
set has roughly zero mean and unit variance. Remember this - Part 1 relies on it.

## Part 1 - Randomizing the data

Every experiment in the paper is one of the following transformations applied to the **training**
set, with nothing else changed (Sec 2.1). Test images and test labels are never corrupted.

In [ ]:
class ArrayDataset(torch.utils.data.Dataset):
    """Serves (image, label) pairs from tensors, optionally augmented."""

    def __init__(self, X, y, augment=False):
        self.X, self.y, self.augment = X, y, augment

    def __len__(self):
        # TODO
        raise NotImplementedError

    def __getitem__(self, i):
        # TODO: return the i-th (image, label). If self.augment, apply U.random_crop_flip
        # to the image (and only to the image) first.
        raise NotImplementedError

In [ ]:
def corrupt_labels(y, p, num_classes=10, seed=0):
    """Independently with probability p, replace a label by a uniformly random class.

    Returns a new tensor; y must not be modified. Draw the randomness from a
    torch.Generator seeded with `seed` so the corruption is reproducible.
    """
    # TODO
    raise NotImplementedError

In [ ]:
for p in (0.0, 0.5, 1.0):
    changed = (corrupt_labels(ytr, p) != ytr).float().mean()
    print(f"p={p}: {changed:.3f} of labels actually changed")

At `p=1.0` only ~90% of the labels change. Make sure you understand why before moving on.

In [ ]:
def randomize_inputs(X, mode, seed=0):
    """Randomize the images of X (N,3,32,32); returns a new tensor.

    'shuffled_pixels' one random permutation of the 32*32 pixel positions, applied to every
                      image and identically to all three channels. The permutation must depend
                      only on `seed`, so that calling this on the train and the test set with the
                      same seed applies the *same* permutation, as in the paper.
    'random_pixels'   an independent pixel permutation for each image.
    'gaussian'        i.i.d. N(0, 1) pixels. X is standardized, so this matches the mean and
                      variance of the dataset, which is what the paper requires.
    """
    # TODO
    raise NotImplementedError

In [ ]:
U.show_images(Xtr, "original")
for mode in ("shuffled_pixels", "random_pixels", "gaussian"):
    U.show_images(randomize_inputs(Xtr[:8], mode), mode)

## Part 2 - Models

Two architectures from the paper's CIFAR-10 experiments: an MLP (the paper's `MLP 1x512`) and a
small convolutional network standing in for its `Inception (small)`.

In [ ]:
class MLP(nn.Module):
    """Flatten, then Linear -> ReLU for each width in `hidden`, then a linear classifier."""

    def __init__(self, hidden=(512,), in_dim=3 * 32 * 32, num_classes=10):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError

In [ ]:
class SmallCNN(nn.Module):
    """For each width w in `widths`: Conv3x3(padding=1) -> [BatchNorm] -> ReLU -> MaxPool2.
    Then flatten, [Dropout], and a linear classifier.

    Use bias=False in a convolution that is followed by batch norm (the norm's shift makes it
    redundant). With three pooling stages a 32x32 input leaves a 4x4 feature map.
    """

    def __init__(self, widths=(64, 128, 256), use_bn=True, dropout=0.0, num_classes=10):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError

In [ ]:
for name, model in [("MLP 1x512", MLP()), ("MLP 3x512", MLP((512,) * 3)),
                    ("SmallCNN", SmallCNN()), ("SmallCNN w/o BN", SmallCNN(use_bn=False))]:
    print(f"{name:<16} {U.count_params(model):>9,d} params"
          f"  ({U.count_params(model) / N_TRAIN:.0f} per training example)")
assert SmallCNN()(Xtr[:2]).shape == (2, 10) and MLP()(Xtr[:2]).shape == (2, 10)

## Part 3 - The training loop

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    """One pass over `loader`: forward, cross-entropy loss, backward, optimizer step."""
    # TODO
    raise NotImplementedError

In [ ]:
def evaluate(model, loader, device):
    """Returns (mean loss, accuracy) over `loader`.

    Do not build a graph here, and put the model in the mode that makes batch norm and dropout
    behave the way they should at test time.
    """
    # TODO
    raise NotImplementedError

The driver below is given. `run(**cfg)` builds the (possibly randomized) dataset, trains
it, and caches the history under `results/<slug>.json`. Training accuracy is measured after each
epoch in eval mode on the *un-augmented* training set, so it is comparable across conditions.
(The paper used SGD with momentum and a 0.95-per-epoch decay; at this scale that recipe needs
several times more steps than a homework budget allows, so we use Adam with a 0.98-per-epoch
decay. Nothing about the conclusions depends on the choice.)

Note what is *not* in the config: nothing about the architecture changes between the true-label
and random-label experiments.

In [ ]:
DEFAULTS = dict(model="cnn", p=0.0, input_mode=None, augment=False, weight_decay=0.0,
                dropout=0.0, use_bn=True, epochs=EPOCHS, lr=None, seed=0)


def slug(cfg):
    changed = [f"{k}={cfg[k]}" for k in sorted(DEFAULTS) if k != "model" and cfg[k] != DEFAULTS[k]]
    return "_".join([cfg["model"]] + changed)


def fit(model, train_ds, eval_ds, test_ds, epochs, lr, weight_decay, batch_size=128):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.98)
    loaders = dict(train=DataLoader(eval_ds, batch_size=1000),
                   test=DataLoader(test_ds, batch_size=1000))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    hist = {k: [] for k in ("train_loss", "train_acc", "test_loss", "test_acc")}
    t0 = time.time()
    for ep in range(epochs):
        train_one_epoch(model, train_loader, opt, DEVICE)
        sched.step()
        for split, loader in loaders.items():
            loss, acc = evaluate(model, loader, DEVICE)
            hist[f"{split}_loss"].append(loss)
            hist[f"{split}_acc"].append(acc)
        if ep % 20 == 19:
            print(f"    epoch {ep + 1:3d}  train_acc {hist['train_acc'][-1]:.3f}"
                  f"  test_acc {hist['test_acc'][-1]:.3f}")
    hist["seconds"] = round(time.time() - t0, 1)
    return hist


def run(**kwargs):
    cfg = {**DEFAULTS, **kwargs}
    path = RESULTS / f"{slug(cfg)}.json"
    if path.exists():
        return U.load_json(path)
    print(f"running {slug(cfg)}")
    U.set_seed(cfg["seed"])
    y_train = corrupt_labels(ytr, cfg["p"], seed=cfg["seed"])
    X_train, X_test = Xtr, Xte
    if cfg["input_mode"] is not None:
        # shuffled_pixels must use one permutation for both splits; the others are independent
        test_seed = cfg["seed"] if cfg["input_mode"] == "shuffled_pixels" else cfg["seed"] + 1
        X_train = randomize_inputs(Xtr, cfg["input_mode"], seed=cfg["seed"])
        X_test = randomize_inputs(Xte, cfg["input_mode"], seed=test_seed)
    if cfg["model"] == "cnn":
        model = SmallCNN(use_bn=cfg["use_bn"], dropout=cfg["dropout"])
    else:
        model = MLP((512,))
    hist = fit(model,
               ArrayDataset(X_train, y_train, augment=cfg["augment"]),
               ArrayDataset(X_train, y_train),
               ArrayDataset(X_test, yte),
               epochs=cfg["epochs"],
               lr=cfg["lr"] or 1e-3,
               weight_decay=cfg["weight_decay"])
    hist["config"] = cfg
    U.save_json(hist, path)
    return hist

## Part 4 - Fitting random labels and random pixels

Figure 1a of the paper. Same network, same optimizer, same epoch budget for all five conditions;
only the training data differs. This cell does the bulk of the compute - expect ~5 minutes the
first time.

In [ ]:
CONDITIONS = {
    "true labels": dict(),
    "random labels": dict(p=1.0),
    "shuffled pixels": dict(input_mode="shuffled_pixels"),
    "random pixels": dict(input_mode="random_pixels"),
    "gaussian": dict(input_mode="gaussian"),
}
cnn = {k: run(model="cnn", **cfg) for k, cfg in CONDITIONS.items()}
mlp = {k: run(model="mlp", **cfg) for k, cfg in CONDITIONS.items()}

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex=True)
for row, (name, hists) in enumerate([("SmallCNN", cnn), ("MLP 1x512", mlp)]):
    U.plot_curves(hists, "train_loss", "training loss", axes[row, 0])
    U.plot_curves(hists, "train_acc", "training accuracy", axes[row, 1])
    U.plot_curves(hists, "test_acc", "test accuracy", axes[row, 2])
    axes[row, 0].set_ylabel(f"{name}\ntraining loss")
    axes[row, 2].axhline(0.1, ls=":", c="k")
fig.tight_layout()

In [ ]:
U.print_table(
    ["condition", "CNN train", "CNN test", "MLP train", "MLP test"],
    [[k, f"{cnn[k]['train_acc'][-1]:.3f}", f"{cnn[k]['test_acc'][-1]:.3f}",
      f"{mlp[k]['train_acc'][-1]:.3f}", f"{mlp[k]['test_acc'][-1]:.3f}"] for k in CONDITIONS])

**Q1.** The CNN reaches essentially 100% training accuracy on completely random labels.
State what this implies for the empirical Rademacher complexity of this architecture on this
training set (Sec 2.2, Eq. 1), and explain why a generalization bound of the form
`test error <= train error + complexity term` is then useless here.

*Your answer:*

**Q2.** (a) `random pixels` and `gaussian` reach zero training loss *sooner* than `random
labels` - compare the epoch at which each first hits 100% training accuracy. Why does destroying
the images make the fitting problem *easier* than destroying the labels? (b) `random pixels`
nevertheless ends well above 10% test accuracy. A different permutation is applied to every image,
so what information about an image survives an arbitrary permutation of its pixels, and why does
that let the network beat chance?

*Your answer:*

**Q3.** Compare the CNN and the MLP under `shuffled pixels`. One loses far less test
accuracy than the other. Which, and what property of the architecture explains it? What does this
say about where the CNN's advantage on natural images actually comes from?

*Your answer:*

## Part 5 - Interpolating between signal and noise

Figures 1b and 1c: sweep the label corruption level `p` from 0 to 1. `p=0.0` and `p=1.0` are
already cached from Part 4.

In [ ]:
PS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
sweep = {p: run(model="cnn", p=p) for p in PS}

In [ ]:
def epochs_to_fit(history, threshold=0.99):
    """First epoch (counting from 1) at which training accuracy reaches `threshold`,
    or float("nan") if it never does."""
    # TODO
    raise NotImplementedError

In [ ]:
base = epochs_to_fit(sweep[0.0])
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
axes[0].plot(PS, [epochs_to_fit(sweep[p]) / base for p in PS], "o-")
axes[0].set_xlabel("label corruption p"); axes[0].set_ylabel("time to overfit (relative)")
axes[1].plot(PS, [1 - sweep[p]["test_acc"][-1] for p in PS], "o-")
axes[1].axhline(0.9, ls=":", c="k")
axes[1].set_xlabel("label corruption p"); axes[1].set_ylabel("test error")
U.plot_curves({f"p={p}": sweep[p] for p in PS}, "train_acc", "training accuracy", axes[2])
for ax in axes[:2]:
    ax.grid(alpha=0.3)
fig.tight_layout()

U.print_table(["p", "epochs to fit", "train acc", "test acc"],
              [[p, epochs_to_fit(sweep[p]), f"{sweep[p]['train_acc'][-1]:.3f}",
                f"{sweep[p]['test_acc'][-1]:.3f}"] for p in PS])

**Q4.** Training accuracy stays at ~100% for every `p`, yet test accuracy degrades smoothly
instead of collapsing as soon as noise appears. What does the shape of that curve tell you about
what the network does with the uncorrupted part of the training set?

*Your answer:*

## Part 6 - Does regularization explain generalization?

Table 1 and Table 4 of the paper, in miniature. Each regularizer is toggled on true labels and on
random labels. The paper's claim is precise and worth testing separately in both directions:
explicit regularization is *neither necessary* for generalization *nor sufficient* to prevent
memorization.

In [ ]:
REGULARIZERS = {
    "none": dict(),
    "weight decay 5e-4": dict(weight_decay=5e-4),
    "augmentation": dict(augment=True),
    "wd + augmentation": dict(weight_decay=5e-4, augment=True),
    "dropout 0.5": dict(dropout=0.5),
    "no batch norm": dict(use_bn=False),
}
true_labels = {k: run(model="cnn", **cfg) for k, cfg in REGULARIZERS.items()}
random_labels = {k: run(model="cnn", p=1.0, **cfg) for k, cfg in REGULARIZERS.items()}

In [ ]:
U.print_table(
    ["regularizer", "true: train", "true: test", "random: train", "random: test"],
    [[k, f"{true_labels[k]['train_acc'][-1]:.3f}", f"{true_labels[k]['test_acc'][-1]:.3f}",
      f"{random_labels[k]['train_acc'][-1]:.3f}", f"{random_labels[k]['test_acc'][-1]:.3f}"]
     for k in REGULARIZERS])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
U.plot_curves(true_labels, "test_acc", "test accuracy", axes[0])
U.plot_curves(random_labels, "train_acc", "training accuracy", axes[1])
axes[0].set_title("true labels: test accuracy")
axes[1].set_title("random labels: training accuracy")
fig.tight_layout()

**Q5.** From your table: (a) is any of these regularizers *necessary* for the CNN to
generalize on true labels? (b) is any of them *sufficient* to prevent memorization of random
labels? (c) Batch norm was not introduced as a regularizer - what did removing it do to test
accuracy, and to the ability to memorize? Quote the numbers you are using.

*Your answer:*

## Part 7 - Finite-sample expressivity (Theorem 1)

The experiments show the network *can* fit random labels. Theorem 1 says we should not have been
surprised: a two-layer ReLU network with only $2n + d$ parameters represents *any* labeling of
*any* $n$ distinct points in $d$ dimensions, and its weights can be written down in closed form -
no gradient descent involved.

$$c(z) \;=\; \sum_{j=1}^{n} w_j \,\mathrm{relu}(\langle a, z\rangle - b_j)$$

The construction: pick $a$ so the projections $x_i = \langle a, z_i\rangle$ are distinct and sort
them; choose thresholds that interleave them, $b_1 < x_1 < b_2 < x_2 < \dots < b_n < x_n$. Then
$A_{ij} = \mathrm{relu}(x_i - b_j)$ is lower triangular with strictly positive diagonal
(Lemma 1), so $Aw = y$ has a unique solution.

In [ ]:
class Memorizer(nn.Module):
    """c(z) = sum_j w_j * relu(<a, z> - b_j).  Exactly 2n + d parameters."""

    def __init__(self, a, b, w):
        super().__init__()
        self.a, self.b, self.w = nn.Parameter(a), nn.Parameter(b), nn.Parameter(w)

    def forward(self, z):
        return torch.relu((z @ self.a)[:, None] - self.b) @ self.w

In [ ]:
def build_memorizer(Z, y):
    """Z: (n, d) distinct points, y: (n,) arbitrary real targets. Returns a Memorizer with
    c(Z[i]) == y[i] for every i, with no training.

    Steps: draw a random `a`; sort the projections; set interleaving thresholds `b`; build the
    lower-triangular A and solve A w = y with torch.linalg.solve_triangular. Watch the ordering -
    permuting the points must permute the targets the same way.
    """
    # TODO
    raise NotImplementedError

In [ ]:
n, d = 1000, 100
g = torch.Generator().manual_seed(0)
Z = torch.randn(n, d, generator=g, dtype=torch.float64)
y = torch.randint(10, (n,), generator=g).double()

net = build_memorizer(Z, y)
print(f"parameters: {U.count_params(net)}  (2n + d = {2 * n + d})")
print(f"max |c(z_i) - y_i| = {(net(Z) - y).abs().max():.2e}")

Now watch the construction degrade, and check it on real images with random labels.

In [ ]:
rows = []
for n in (100, 500, 1000, 2000, 5000):
    Z = torch.randn(n, 100, generator=g, dtype=torch.float64)
    y = torch.randint(10, (n,), generator=g).double()
    net = build_memorizer(Z, y)
    gap = ((Z @ net.a).sort().values - net.b).min()
    rows.append([n, f"{(net(Z) - y).abs().max():.1e}", f"{net.w.abs().max():.1e}", f"{gap:.1e}"])
U.print_table(["n", "max error", "max |w_j|", "min_i (x_i - b_i)"], rows)

n = 2000
Z = Xtr[:n].flatten(1).double()
y = corrupt_labels(ytr[:n], p=1.0).double()
net = build_memorizer(Z, y)
print(f"\nCIFAR-10 images, random labels: n={n}, d={Z.shape[1]}, "
      f"{U.count_params(net)} params, max error {(net(Z) - y).abs().max():.2e}")

**Q6.** As `n` grows, the error and `max |w_j|` blow up. Explain this using the smallest
diagonal entry of `A` (Lemma 1), and say what the theorem therefore does *not* claim.

*Your answer:*

## Part 8 - Wrap-up

**Q7.** Both your true-label and your random-label runs reach ~100% training accuracy, so nothing
measured on the training loss can separate them. Propose one quantity you could measure at the
*end* of training that you would expect to differ between the two, and describe the experiment -
including its control - that would test whether that quantity predicts test accuracy. One
paragraph.

*Your answer:*